# BRFSS 2015 — AUDIT ONLY
User reports 441456 XPT rows. This notebook checks actual evidence; no Train/CV or Final Test is invoked. Unknown codebook/raw-code/split-history requirements remain pending.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json
from datetime import datetime, timezone
CHECKOUT=Path('/content/PPTK')
if not CHECKOUT.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/khanhleviet2006/PPTK.git',str(CHECKOUT)],check=True)
else:
    changes=subprocess.run(['git','-C',str(CHECKOUT),'status','--porcelain'],check=True,text=True,capture_output=True).stdout
    if not changes.strip():
        subprocess.run(['git','-C',str(CHECKOUT),'fetch','origin','main'],check=True)
        subprocess.run(['git','-C',str(CHECKOUT),'merge','--ff-only','FETCH_HEAD'],check=True)
    else:
        print('Local checkout edits preserved; no update/reset attempted.')
PROJECT=CHECKOUT/'BRFSS2015_Improvement'
assert (PROJECT/'src/audit_review.py').is_file(), 'Updated audit workflow is required; local edits were preserved.'
assert (PROJECT/'src/data_io.py').is_file(), 'Use the updated project checkout; existing files are never reset automatically.'
DATA=Path('/content/drive/MyDrive/Diabetes_Statistics_2015/01_Raw_Data/LLCP2015XPT.zip')
assert DATA.is_file(), f'ZIP not found: {DATA}'
RUNS=Path('/content/drive/MyDrive/BRFSS2015_Improvement_Runs')
RUNS.mkdir(parents=True,exist_ok=True)
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
# Copy the partial schema into Drive only when it does not already exist.
# Review edits persist here across Colab disconnects; never overwrite them on Run All.
SCHEMA=RUNS/'schema.reviewed.json'
AUDIT_ONLY=True
if not SCHEMA.exists():
    with SCHEMA.open('x',encoding='utf8') as f:
        f.write((PROJECT/'schema.reviewed.json').read_text(encoding='utf8'))
PREFLIGHT=RUNS/f'preflight_{STAMP}.json'
OUTPUT=RUNS/f'run_{STAMP}'
print('DATA:',DATA,'\nSCHEMA:',SCHEMA,'\nOUTPUT:',OUTPUT)
print('Existing schema preserved:',SCHEMA.is_file())

print('Kernel Python:',sys.version)
sys.path.insert(0,str(PROJECT/'src'))
from notebook_runtime import run_logged, new_artifact


In [ ]:
run_logged([sys.executable,'-m','pip','install','-r',str(PROJECT/'requirements-audit.txt')],RUNS,'audit_install')
run_logged([sys.executable,str(PROJECT/'tests/test_audit_runtime.py')],RUNS,'audit_runtime_tests')


## ZIP and XPT preflight — no fitting and no final evaluation
The reader accepts exactly one XPT member, refuses unsafe names/links/encryption/ambiguous archives, bounds decompressed size (4 GiB) and compression ratio (200), and reads to EOF to check CRC. It uses a disk-backed temporary file, removed on success or failure. It never extracts archive paths or writes the raw XPT to Drive. Ensure Colab has enough temporary disk for the full decompressed member and RAM for later training.

This step reports header metadata plus at most 1,000 sample records internally; it does not certify the entire raw domain. It performs no Train/Test split, preprocessing fit, model selection or scoring. A ZIP SHA256 establishes identity, not authenticity without a trusted reference hash.


In [ ]:
# A fresh artifact per execution prevents FileExistsError when rerunning this cell.
PREFLIGHT=new_artifact(RUNS,'preflight')
run_logged([sys.executable,str(PROJECT/'src/audit_cli.py'),'inspect','--data',str(DATA),'--schema',str(SCHEMA),'--output',str(PREFLIGHT)],RUNS,'inspect')
preflight=json.loads(PREFLIGHT.read_text())
print('XPT member:',preflight.get('member'))
print('Reported rows:',preflight['xpt_reported_rows'])
print('Missing required columns:',preflight['missing_required_columns'])
# Missing schema columns are audit findings, not reasons to prevent evidence gathering.
sys.path.insert(0,str(PROJECT/'src'))
from audit_review import migrate_bp_schema
migration=migrate_bp_schema(SCHEMA,preflight['columns'])
print('Evidence-based schema migration:',migration)
schema=json.loads(SCHEMA.read_text())
required=set(schema['features'])|{schema['target']}
print('Remaining predictor/target columns absent:',sorted(required-set(preflight['columns'])))


## Full raw-code audit — no learning or model evaluation
This scans the whole XPT to count raw codes for the predefined candidates and target. It can use substantial RAM. Counts do not establish code meanings; compare them against the official codebook. No split, fitted preprocessing, feature selection or model scoring occurs. Original data and earlier audits are preserved.

In [ ]:
# Preserve old audits. Reuse only a provenance-matching, complete audit including design diagnostics.
from audit_review import sha256
import pandas as pd
needed=set(schema['features'])|{schema['target']}|set(schema.get('alternatives',{}))
RAW_AUDIT=None
for candidate in sorted(RUNS.glob('raw_codes_*/raw_code_audit.json'),reverse=True):
    meta=json.loads(candidate.read_text());counts=candidate.parent/'raw_code_counts.csv'
    if (counts.is_file() and meta.get('container_sha256')==preflight['container_sha256']
        and meta.get('survey_design_audit') and meta.get('raw_code_counts_sha256')==sha256(counts)):
        present=set(pd.read_csv(counts,usecols=['variable']).variable)
        if needed <= present:
            RAW_AUDIT=candidate.parent;print('Reusing matched audit:',RAW_AUDIT);break
if RAW_AUDIT is None:
    RAW_AUDIT=new_artifact(RUNS,'raw_codes','')
    run_logged([sys.executable,str(PROJECT/'src/audit_cli.py'),'audit-codes','--data',str(DATA),'--schema',str(SCHEMA),'--output',str(RAW_AUDIT)],RUNS,'raw_audit')
print('Full raw-code frequencies:',RAW_AUDIT/'raw_code_counts.csv')


## Official CDC source — retrieve evidence without approving the schema
The URLs below are intended CDC references; retrieval failed in the cloud development environment. This cell attempts retrieval from Colab using normal TLS verification. It stores a hash and access metadata for review, but **never changes review flags**. Read the official variable entries and calculated-variable definitions, record document version and actual page/table citations, and verify every missing/raw-domain/recode/scale decision in the Drive schema.


In [ ]:
import urllib.request, urllib.parse, hashlib
CODEBOOK_URL='https://www.cdc.gov/brfss/annual_data/2015/pdf/codebook15_llcp.pdf'
try:
    request=urllib.request.Request(CODEBOOK_URL,headers={'User-Agent':'BRFSS2015-Research/1.0'})
    with urllib.request.urlopen(request,timeout=60) as response:
        host=urllib.parse.urlparse(response.url).hostname or ''
        if host != 'cdc.gov' and not host.endswith('.cdc.gov'):
            raise ValueError('Unexpected non-CDC source redirect')
        pdf=response.read(25*1024**2+1)
        if len(pdf)>25*1024**2 or not pdf.startswith(b'%PDF-'):
            raise ValueError('Response is too large or is not a PDF; inspect the official annual landing page')
    sha=hashlib.sha256(pdf).hexdigest()
    pdf_path=RUNS/f'codebook15_llcp_{sha[:16]}.pdf'
    if not pdf_path.exists():
        with pdf_path.open('xb') as f:f.write(pdf)
    metadata={'requested_url':CODEBOOK_URL,'resolved_url':response.url,'sha256':sha,'bytes':len(pdf),'retrieved_at_utc':datetime.now(timezone.utc).isoformat(),'review_complete':False}
    metadata_path=new_artifact(RUNS,'codebook_access')
    with metadata_path.open('x') as f:json.dump(metadata,f,indent=2)
    print('Retrieved evidence for manual review:',pdf_path,metadata)
    from audit_review import extract_codebook_evidence
    evidence_path=new_artifact(RUNS,'codebook_page_evidence')
    evidence=extract_codebook_evidence(pdf_path,SCHEMA,evidence_path)
    print('Actual PDF page candidates:',{k:[p['pdf_page_1based'] for p in v] for k,v in evidence['candidate_pages'].items()})
    print('Review page text and meanings in:',evidence_path)
    print('Page matches do not certify codes, year, timing, or grant VERIFIED.')
except Exception as exc:
    print('Codebook remains UNVERIFIED:',type(exc).__name__,str(exc))
    print('Official landing page: https://www.cdc.gov/brfss/annual_data/annual_2015.html')


## Evidence review — pending items are reported, not silently approved
The following existing cell compares all raw codes, integrity metadata and survey diagnostics, and lists unresolved citations/timing/history. It does not clear any requirement or set review flags. Schema migration preserves the old Drive file in a content-addressed backup.

In [ ]:
import importlib
import audit_review as audit_review_module
importlib.invalidate_caches()
importlib.reload(audit_review_module)
print("Evidence Review module:",audit_review_module.__file__)
from audit_review import compare_raw_codes, readiness, sha256
# Read the current Drive schema, not the earlier in-memory preflight copy.
schema=json.loads(SCHEMA.read_text(encoding="utf-8"))
print("Schema used:",SCHEMA.resolve(),"SHA256:",sha256(SCHEMA))
print("Current target review:",schema.get("target_review",{}))
REVIEW=new_artifact(RUNS,'evidence_review','')
summary=compare_raw_codes(SCHEMA,RAW_AUDIT/'raw_code_counts.csv',RAW_AUDIT/'raw_code_audit.json',PREFLIGHT,REVIEW)
display(pd.read_csv(REVIEW/'variable_evidence_review.csv'))
print('Integrity:',json.dumps(summary['integrity'],indent=2))
print('Survey diagnostics:',json.dumps(summary['survey_design'],indent=2))
print('Split/Test history:',json.dumps(summary['split_history'],indent=2))
print('Evidence already recorded:',json.dumps(summary['evidence_available'],indent=2))
print('Computed unresolved checks:',json.dumps(summary['computed_issues'],indent=2))
print('Declared requirements retained for reconciliation:',json.dumps(summary['declared_unverified_items'],indent=2))
print('AUDIT COMPLETE; REVIEW MAY REMAIN PENDING. Train/CV and Final Test were not run.')
print('Reports:',REVIEW)

# Locate possible prior splits without reading, changing or selecting one.
known_roots=[RUNS,Path('/content/drive/MyDrive/Diabetes_Statistics_2015'),Path('/content/drive/MyDrive/BRFSS2015_Research')]
prior_splits=[]
for root in known_roots:
    if root.exists():
        for path in root.rglob('split.npz'):
            prior_splits.append(str(path))
            if len(prior_splits)>=50:break
    if len(prior_splits)>=50:break
print('Possible earlier split files (up to 50, not validated):',prior_splits)
print('Prior Test-use history cannot be inferred from the existence or absence of these files.')


## Train/CV — DISABLED IN THIS PHASE
The established split/CV protocol remains in the source, but there is no executable training cell in this audit-only notebook. Resolve evidence and obtain authorization for the next phase first.

In [ ]:
print('Train/CV: DISABLED. Final Test: LOCKED. No models fitted.')


In [ ]:
print('No model metrics are produced in the audit phase.')


## Final Test — LOCKED
There is deliberately no executable final-evaluation cell. The CLI also refuses `final` without explicit `--unlock-final`; the existing one-time evaluation marker and frozen-artifact hashes still apply. Unlocking belongs to a separate step after data/protocol review and final configuration selection. Do not use Test to choose features, models, thresholds or hypotheses. Comparison to Chang et al. (2022) requires verification of an equivalent cohort/label/evaluation protocol; exceeding a point benchmark alone is insufficient.
